In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

# Load cleaned data
df = pd.read_csv('appointments_clean.csv')

# ==========================================
# Task 3.1 — Features, split and baseline
# ==========================================
X = df[['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent']].copy()
X['reminder_sent'] = X['reminder_sent'].map({'Yes': 1, 'No': 0})
dummies = pd.get_dummies(df[['booking_channel', 'visit_type']], drop_first=False)
X = pd.concat([X, dummies], axis=1)

y = df['no_show']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X has {X.shape[1]} columns; Train rows: {len(X_train)}, Test rows: {len(X_test)}")
print(f"Test no-shows: {y_test.sum()} ({y_test.mean()*100:.1f}%)")
baseline_acc = (y_test == 0).mean()
print(f"Baseline accuracy: {baseline_acc:.4f} ({ (y_test == 0).sum() } of {len(y_test)})")


# ==========================================
# Task 3.2 — Logistic regression
# ==========================================
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)

y_pred_lr = lr.predict(X_test_s)
y_prob_lr = lr.predict_proba(X_test_s)[:, 1]

print("\n--- Task 3.2 Results ---")
print("Accuracy:", round(accuracy_score(y_test, y_pred_lr), 3))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("Precision:", round(precision_score(y_test, y_pred_lr), 2))
print("Recall:", round(recall_score(y_test, y_pred_lr), 2))
print("F1:", round(f1_score(y_test, y_pred_lr), 2))
print("ROC-AUC:", round(roc_auc_score(y_test, y_prob_lr), 3))

reminder_idx = list(X.columns).index('reminder_sent')
c_rem = lr.coef_[0][reminder_idx]
s_rem = scaler.scale_[reminder_idx]
print(f"reminder_sent coef: {c_rem:.2f}")
print(f"np.exp(coef): {np.exp(c_rem):.2f}")
print(f"np.exp(coef / scale) for real No->Yes change: {np.exp(c_rem / s_rem):.2f}")


# ==========================================
# Task 3.3 — Decision tree and overfitting
# ==========================================
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train, y_train)
print("\n--- Task 3.3 Results ---")
print("Full-depth tree train acc:", round(accuracy_score(y_train, dt_full.predict(X_train)), 2),
      "| test acc:", round(accuracy_score(y_test, dt_full.predict(X_test)), 2), f"(depth {dt_full.get_depth()})")

dt_3 = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_3.fit(X_train, y_train)
print("Depth-3 tree train acc:", round(accuracy_score(y_train, dt_3.predict(X_train)), 2),
      "| test acc:", round(accuracy_score(y_test, dt_3.predict(X_test)), 2))


# ==========================================
# Task 3.4 — Random forest and comparison table
# ==========================================
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print("\n--- Task 3.4 Results ---")
print("RF accuracy:", round(accuracy_score(y_test, y_pred_rf), 2))
print("RF recall:", round(recall_score(y_test, y_pred_rf), 2))
print("RF ROC-AUC:", round(roc_auc_score(y_test, y_prob_rf), 2))
top_importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(3)
print("Top 3 feature importances:\n", top_importances)


# ==========================================
# Task 3.5 — Tune, choose, threshold & test
# ==========================================
# (a) GridSearchCV
grid = GridSearchCV(RandomForestClassifier(random_state=42), {"n_estimators": [100, 200], "max_depth": [3, 5, 8]}, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("\n--- Task 3.5 Results ---")
print("Best params:", grid.best_params_)
print("Best score (CV ROC-AUC):", round(grid.best_score_, 2))

# (b) Cross-validated training ROC-AUC comparison
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_auc_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_dt3 = cross_val_score(dt_3, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_auc_rf = cross_val_score(rf, X_train, y_train, cv=5, scoring="roc_auc").mean()

print(f"CV Training ROC-AUC -> Logistic Regression: {cv_auc_lr:.3f}, Depth-3 Tree: {cv_auc_dt3:.3f}, Random Forest: {cv_auc_rf:.3f}")

# (c) Out-of-fold probability evaluation on training set
oof_lr = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
for t in [0.5, 0.35]:
    preds = (oof_lr >= t).astype(int)
    print(f"Threshold {t} -> Precision: {precision_score(y_train, preds, zero_division=0):.2f}, "
          f"Recall: {recall_score(y_train, preds, zero_division=0):.2f}, "
          f"F1: {f1_score(y_train, preds, zero_division=0):.2f}, Flagged: {preds.sum()}")

# (d) Test evaluation of Logistic Regression at threshold 0.35
lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_preds_35 = (test_probs >= 0.35).astype(int)

print(f"Test Evaluation at threshold 0.35 -> Precision: {precision_score(y_test, test_preds_35):.2f}, "
      f"Recall: {recall_score(y_test, test_preds_35):.2f}, "
      f"Flagged: {test_preds_35.sum()}, ROC-AUC: {roc_auc_score(y_test, test_probs):.2f}")

# (e) Save test_risk_scores.csv
test_results = pd.DataFrame({
    'appointment_id': df.loc[X_test.index, 'appointment_id'],
    'no_show': y_test,
    'risk_prob': test_probs,
    'flag': test_preds_35
})
test_results.to_csv('test_risk_scores.csv', index=False)
print("Saved test_risk_scores.csv successfully with", len(test_results), "rows.")